<a href="https://colab.research.google.com/github/Abel-Kurian/Ai-internship/blob/main/Day7_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv("/content/IMDB Dataset.csv",encoding='latin1')
print(df.head())
print(df["sentiment"].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [ ]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [ ]:
df["sentiment"] = df["sentiment"].map({"positive":1,"negative":0})

In [ ]:
import re
negation_words = ["not good","not bad","not great","dont like","didn't like","never liked","wasn't good","no good","not great"]

def clean_text(text):
  text = text.lower()

  text = re.sub(f"[^a-zA-Z\0']"," ",text)

  #convert negations into tokens
  for phrase in negation_words:
    text=text.replace(phrase,phrase.replace(" ","_"))

  return text

In [ ]:
df["review"]=df["review"].apply(clean_text)

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(df["review"],df["sentiment"],test_size=0.2,random_state=42)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 20000
max_len = 250

tokenizer = Tokenizer(num_words=vocab_size,oov_token="<OOV>") #oov - out of vocabolary(to predict things out of 20000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq,maxlen=max_len,padding="post")
X_test_pad = pad_sequences(X_test_seq,maxlen=max_len,padding="post")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout

model=Sequential([
    Embedding(vocab_size,128,input_length=max_len),
    LSTM(128,dropout=0.3,recurrent_dropout=0.3),
    Dense(64,activation="relu"),
    Dropout(0.3),
    Dense(1,activation="sigmoid")
])

In [ ]:
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])


In [ ]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(X_train_pad,y_train,epochs=5,batch_size=64,validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 415s 822ms/step - accuracy: 0.5524 - loss: 0.6673 - val_accuracy: 0.6049 - val_loss: 0.6222
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 412s 825ms/step - accuracy: 0.6011 - loss: 0.6103 - val_accuracy: 0.5995 - val_loss: 0.6285
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 440s 821ms/step - accuracy: 0.7174 - loss: 0.5223 - val_accuracy: 0.8159 - val_loss: 0.4217
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 482s 902ms/step - accuracy: 0.8576 - loss: 0.3546 - val_accuracy: 0.8668 - val_loss: 0.3366
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 469s 836ms/step - accuracy: 0.9054 - loss: 0.2490 - val_accuracy: 0.8708 - val_loss: 0.3262


In [ ]:
loss, acc = model.evaluate(X_test_pad, y_test)
print("Test Accuracy:", acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 34s 105ms/step - accuracy: 0.8751 - loss: 0.3153
Test Accuracy: 0.8751000165939331


In [ ]:
def predict_sentiment(review):

  review = clean_text(review)

  seq = tokenizer.texts_to_sequences([review])

  padded = pad_sequences(seq,maxlen=max_len,padding="post")

  pred = model.predict(padded)[0][0]

  return "Positive" if pred>=0.5 else "Negative"

In [ ]:
predict_sentiment("bad")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step


'Negative'